<a href="https://colab.research.google.com/github/riyarvv/End-to-End-Customer-Churn-ML-Pipeline/blob/main/notebooks/15_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End Customer Churn ML Pipeline

## Notebook 15 - Hyperparameter Tuning

### Objective

Optimize machine learning models using GridSearchCV and RandomizedSearchCV to improve predictive performance.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import pandas as pd
import numpy as mp
import joblib
import sys

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score

In [21]:
project_path="/content/drive/MyDrive/ML Projects/End-to-End-Customer-Churn-ML-Pipeline"

if project_path not in sys.path:
    sys.path.append(project_path)

In [22]:
from src.data_loader import load_data
from src.preprocessing import create_preprocessor

In [23]:
X_train = load_data(project_path+"/data/processed/X_train.csv")
X_test = load_data(project_path+"/data/processed/X_test.csv")

y_train = load_data(project_path+"/data/processed/y_train.csv")
y_test = load_data(project_path+"/data/processed/y_test.csv")

In [24]:
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numerical_features = X_train.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = create_preprocessor(categorical_features, numerical_features)

In [25]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

We are tuning only Random Forest Classifier because it has the most number of hyperparameters. Logistic Regression has far fewer important hyperparameters, so it's a good baseline but less interesting for learning tuning.

In [26]:
param_grid = {
    "model__n_estimators": [50, 100, 200],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10]
}

In [30]:
from sklearn.metrics import make_scorer, f1_score

f1_scorer = make_scorer(
    f1_score,
    pos_label="Yes"
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring=f1_scorer,
    n_jobs=-1
)

In [31]:
grid_search.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['gender',
                                                                          'Partner',
                                                                          'Dependents',
                                                                          'PhoneService',
                                                                          'MultipleLines',
                                                                          'InternetService',
                                                                          'OnlineSecurity',
                                                                          'OnlineBackup',
                                                                          'DeviceProtection',
                                                                          'TechSupport',
                                                                          'StreamingTV',
                                                                          'StreamingMovies',
                                                                          'Contract',
                                                                          'PaperlessBilling',
                                                                          'Pa...
                                                                         StandardScaler(),
                                                                         ['SeniorCitizen',
                                                                          'tenure',
                                                                          'MonthlyCharges',
                                                                          'TotalCharges',
                                                                          'NewCustomer',
                                                                          'HighMonthlyCharges'])])),
                                       ('model',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [5, 10, 15, None],
                         'model__min_samples_split': [2, 5, 10],
                         'model__n_estimators': [50, 100, 200]},
             scoring=make_scorer(f1_score, response_method='predict', pos_label=Yes))

In [32]:
print(grid_search.best_params_)

{'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [33]:
print(grid_search.best_score_)

0.581128620830869


In [34]:
best_model = grid_search.best_estimator_

In [35]:
predictions = best_model.predict(X_test)

In [37]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print(accuracy)

0.7924662402274343


In [40]:
logistic = joblib.load(project_path+"/models/logistic_regression.pkl")
tree = joblib.load(project_path+"/models/decision_tree.pkl")
forest = joblib.load(project_path+"/models/random_forest.pkl")

In [41]:
models = {
    "Logistic Regression": logistic,
    "Decision Tree": tree,
    "Random Forest": forest,
    "Tuned Random Forest": best_model
}

In [42]:
from src.model_utils import evaluate_model

results =[]

for name, model in models.items():
    metrics = evaluate_model(
        model,
        X_test,
        y_test
    )

    metrics["Model"]=name

    results.append(metrics)

In [43]:
results = pd.DataFrame(results)

print(results)

   Accuracy  Precision    Recall  F1 Score   ROC AUC                Model
0  0.792466   0.632258  0.524064  0.573099  0.834850  Logistic Regression
1  0.724236   0.482051  0.502674  0.492147  0.653082        Decision Tree
2  0.778252   0.603333  0.483957  0.537092  0.814540        Random Forest
3  0.792466   0.635762  0.513369  0.568047  0.831392  Tuned Random Forest


In [44]:
joblib.dump(
    best_model,
    project_path+"/models/best_random_forest.pkl"
)

['/content/drive/MyDrive/ML Projects/End-to-End-Customer-Churn-ML-Pipeline/models/best_random_forest.pkl']

In [45]:
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(handle_unknown='ignore'),
                                                                               ['gender',
                                                                                'Partner',
                                                                                'Dependents',
                                                                                'PhoneService',
                                                                                'MultipleLines',
                                                                                'InternetService',
                                                                                'OnlineSecurity',
                                                                                'OnlineBackup',
                                                                                'DeviceProtection',
                                                                                'TechSupport',
                                                                                'StreamingTV',
                                                                                'StreamingMovies',
                                                                                'Contract',
                                                                                'PaperlessBillin...
                                                                                'TenureGroup']),
                                                                              ('num',
                                                                               StandardScaler(),
                                                                               ['SeniorCitizen',
                                                                                'tenure',
                                                                                'MonthlyCharges',
                                                                                'TotalCharges',
                                                                                'NewCustomer',
                                                                                'HighMonthlyCharges'])])),
                                             ('model',
                                              RandomForestClassifier(random_state=42))]),
                   n_jobs=-1,
                   param_distributions={'model__max_depth': [5, 10, 15, None],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [50, 100, 200]},
                   random_state=42, scoring='f1')

In [46]:
best_model = random_search.best_estimator_
predictions = best_model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(accuracy)

0.7889125799573561


1. Parameter vs Hyperparameter  
Parameters are learned automatically by the model from the data during training, while hyperparameters are set manually by the engineer before training begins to control the learning process

2. Why do we use Cross Validation?  
To evaluate how well a machine learning model will generalize to new, unseen data

3. Why did we optimize using the F1 score instead of Accuracy?  
To handle imbalanced data, penalize false predictions, and balance precision and recall

4. Why do we write model__max_depth instead of just max_depth?  
We write model__max_depth instead of max_depth to target a specific step's parameter inside a combined machine learning workflow, such as a Scikit-learn Pipeline or GridSearchCV. The double underscore acts as a separator to tell the tool which named step—like a model named model—owns that specific max_depth setting.

5. When would you use RandomizedSearchCV instead of GridSearchCV?  
When your hyperparameter search space is large, your training data is massive, or your computational resources and time are limited.